In [1]:
%load_ext autoreload
%autoreload 2

# coding

> Representing library sequences and motifs as integers, so that scoring a motif is arithmetic
> rather than string matching.

In [2]:
#| default_exp coding

In [3]:
#| hide
from nbdev.showdoc import *
from fastcore.test import test_eq, test_fail
import pandas as pd   # the worked examples only; the module itself needs no pandas

In [4]:
#| export
import functools
import numpy as np
from collections import defaultdict

## Why the scan is arithmetic

The scan in `00_core` has to score **every** motif of every pattern against every sequence, which
is where its runtime goes. Done on strings it is slow and memory-hungry; done on integers it is a
handful of vectorised passes.

A motif of pattern `(d1, spacer, d2)` is pinned down entirely by its `d1 + d2` defined bases — the
spacer is `N` by construction and carries no information — so it can be held as an integer in
`[0, 4**(d1+d2))`. Every per-motif statistic then becomes an `np.bincount` over those integers,
and the library is encoded once and reused for every pattern.

Everything here is internal machinery. Nothing in it decides anything biological; its only job is
to give exactly the answer the readable implementation would, faster. That is what the proofs at
the end of this notebook, and `tests/test_reference_scan.py`, are for.

### The base alphabet, and why it is `ACGT` in that order

`A=0, C=1, G=2, T=3`, and anything else — `N`, lowercase junk, the padding of a short
sequence — is `4`.

The order is not arbitrary. Complementary bases sit at opposite ends of it, so **complementing
a base is `3 - code`**: A↔T is 0↔3, C↔G is 1↔2. That is the whole reverse-complement
implementation, and it is why `encode_library` can return the reverse strand for the price of
one subtraction and one reversal.

Code `4` does double duty: it marks a base that is not ACGT *and* the padding that squares off a
ragged library into a rectangular array. Both must be excluded from the scan, and because they
share a marker, one test excludes both — there is no separate length bookkeeping to get wrong.

In [5]:
#| export
_BASE_CODES = np.full(256, 4, dtype=np.uint8)  # 4 = anything that is not A/C/G/T
for _i, _b in enumerate("ACGT"):
    _BASE_CODES[ord(_b)] = _BASE_CODES[ord(_b.lower())] = _i
_CODE_BASES = np.frombuffer(b"ACGT", dtype=np.uint8)

def encode_library(sequences):
    """
    Encodes DNA sequences as 2-bit base codes for `get_pattern_scores`.

    Args:
        sequences (list): DNA strings (typically library sequences with context);
            they may have different lengths.

    Returns:
        tuple: (fwd, rc) uint8 arrays of shape (n_sequences, max_length) holding the
            base codes of each sequence and of its reverse complement. A=0, C=1, G=2,
            T=3; non-ACGT characters and the padding of shorter sequences are 4.
    """
    max_len = max(len(s) for s in sequences)
    fwd = np.full((len(sequences), max_len), 4, dtype=np.uint8)
    rc = np.full_like(fwd, 4)
    for i, s in enumerate(sequences):
        codes = _BASE_CODES[np.frombuffer(s.encode(), dtype=np.uint8)]
        fwd[i, :len(s)] = codes
        rc[i, :len(s)] = np.where(codes < 4, 3 - codes, 4)[::-1]  # A<->T, C<->G
    return fwd, rc

In [6]:
fwd, rc = encode_library(["ACGTT", "ggN"])
test_eq(fwd, np.array([[0, 1, 2, 3, 3], [2, 2, 4, 4, 4]]))
test_eq(rc,  np.array([[0, 0, 1, 2, 3], [4, 1, 1, 4, 4]]))  # AACGT and NCC, padded

Both strands are left-aligned and padded on the right, so row `i` of `rc` is the reverse
complement of sequence `i` occupying its first `len(s)` columns — not a reversal of the padded
row, which would put the padding at the front.

In [7]:
#| hide
# revcomp really is the string operation, and applying it twice is the identity
import random
_COMP = str.maketrans("ACGT", "TGCA")
_B = {b: i for i, b in enumerate("ACGT")}
_rng = random.Random(0)
for _ in range(200):
    s = "".join(_rng.choice("ACGTN") for _ in range(_rng.randint(5, 40)))
    f, r = encode_library([s])
    test_eq(f[0, :len(s)].tolist(), [_B.get(c, 4) for c in s])
    test_eq(r[0, :len(s)].tolist(), [_B.get(c, 4) for c in s.translate(_COMP)[::-1]])
    test_eq(encode_library([s.translate(_COMP)[::-1]])[1][0, :len(s)].tolist(),
            f[0, :len(s)].tolist())          # rc(rc(s)) == s

### From a window to an integer, and back

A motif of pattern `(d1, spacer, d2)` is pinned down entirely by its `d1 + d2` **defined**
bases — the spacer is `N` by construction and carries no information. So write those bases as
the digits of a **base-4 numeral**:

    CAC NNNN GTAT   ->  digits C,A,C,G,T,A,T  ->  1·4⁶ + 0·4⁵ + 1·4⁴ + 2·4³ + 3·4² + 0·4 + 3

`_pattern_window_codes` builds that numeral by Horner's method — `code = code*4 + next_digit`,
one pass over the defined positions in increasing order. The trick that makes it fast is that
each step operates on a **sliding view**, `encoded_seqs[:, j:j+n_windows]`: position `j` of the
window starting at `w` is absolute index `w+j`, so as `w` runs over all windows that column is
exactly the slice above. Every window of every sequence is therefore built in parallel, with one
vector operation per defined base rather than one per window.

**Rejecting windows.** A window containing a non-ACGT base anywhere — spacer included, to match
the `[ATGC]` regex the reference implementation uses — is set to `4**(d1+d2)`, one past the
largest real code. Which windows those are comes from a prefix sum over the `== 4` mask: the
number of bad bases in `[w, w+width)` is `n_bad[w+width] - n_bad[w]`, so all windows are tested
at once. Padding is code 4, so a window running off the end of a short sequence is rejected by
the same test, with nothing extra written for it.

**Decoding** reverses the numeral. Base 4 is two bits, so digit `i` counted from the most
significant is `(code >> 2*(n-1-i)) & 3` — the same digits in the same order the encoder wrote
them. `decode_motif_codes` then drops `N` into the spacer positions.

In [8]:
#| export
def _pattern_window_codes(encoded_seqs, pattern):
    """
    Motif code of every window of every sequence for `pattern` (d1, spacer, d2).
    Returns an int32 array of shape (n_sequences, n_windows); windows containing a
    base coded 4 (non-ACGT or padding) get the out-of-range code 4**(d1+d2).
    """
    d1, spacer, d2 = pattern
    width = d1 + spacer + d2
    # A motif code is a base-4 numeral with d1+d2 digits, and the rejection sentinel is one
    # past the largest of them, so the whole scheme has to fit in the array's dtype. int32
    # holds 15 defined bases; the shipped patterns use at most 8. Checked rather than assumed:
    # on numpy >= 2 the sentinel assignment below happens to raise, but on numpy 1.x the same
    # assignment wraps silently and every count for this pattern would be quietly wrong.
    if 4 ** (d1 + d2) > np.iinfo(np.int32).max:
        raise ValueError(
            f"pattern {pattern} has {d1 + d2} defined bases; the motif code would overflow "
            f"int32 (max 15). Such a pattern is not usable anyway: it would need "
            f"{4 ** (d1 + d2):.3g} counting bins.")
    n_windows = encoded_seqs.shape[1] - width + 1
    codes = np.zeros((encoded_seqs.shape[0], n_windows), dtype=np.int32)
    for j in list(range(d1)) + list(range(d1 + spacer, width)):  # defined positions only
        codes *= 4
        codes += encoded_seqs[:, j:j + n_windows]
    # Like the [ATGC] regex, reject windows with a non-ACGT base anywhere, spacer included.
    n_bad = np.zeros((encoded_seqs.shape[0], encoded_seqs.shape[1] + 1), dtype=np.int32)
    np.cumsum(encoded_seqs == 4, axis=1, out=n_bad[:, 1:])
    codes[n_bad[:, width:] != n_bad[:, :n_windows]] = 4 ** (d1 + d2)
    return codes

def decode_motif_codes(codes, pattern):
    """
    Converts integer motif codes of `pattern` (d1, spacer, d2) back to motif strings
    with 'N' in the spacer, e.g. CACNNNNGTAT for pattern (3, 4, 4).
    """
    d1, spacer, d2 = pattern
    n_defined = d1 + d2
    codes = np.asarray(codes, dtype=np.int64)
    out = np.full((len(codes), d1 + spacer + d2), ord("N"), dtype=np.uint8)
    bases = _CODE_BASES[(codes[:, None] >> (2 * np.arange(n_defined - 1, -1, -1))) & 3]
    out[:, :d1], out[:, d1 + spacer:] = bases[:, :d1], bases[:, d1:]
    return out.view(f"S{out.shape[1]}").ravel().astype(str).tolist()

With pattern `(1, 2, 2)`, `CAAGT` has a single window whose defined bases are C, G, T, i.e. code `1*16 + 2*4 + 3 = 27`. A window touching the `N` is rejected (code `4**3 = 64`):

In [9]:
fwd, rc = encode_library(["CAAGT", "CAAGTN"])

# A=0 C=1 G=2 T=3, so a motif's code is literally its defined bases as a base-4 numeral:
# CNNGT of pattern (1,2,2) has defined bases C,G,T -> digits 1,2,3 -> int("123", 4).
CNNGT = int("123", 4)
REJECTED = 4 ** 3            # the sentinel: one past the largest 3-digit base-4 numeral
test_eq((CNNGT, REJECTED), (27, 64))

test_eq(_pattern_window_codes(fwd, (1, 2, 2)), np.array([[CNNGT, REJECTED],
                                                        [CNNGT, REJECTED]]))
test_eq(decode_motif_codes([CNNGT, int("000", 4)], (1, 2, 2)), ["CNNGT", "ANNAA"])

### What guarantees this is right?

The scan reports a motif string, but everything between the library and that string is integer
arithmetic. Four properties have to hold, or the output is confidently wrong rather than
obviously broken.

**1. Distinct motifs get distinct codes.** A base-4 numeral with a *fixed* number of digits is
unique to its digit string — that is just positional notation — and the digit count is fixed at
`d1+d2` for a given pattern. Two different motifs of the same pattern therefore cannot collide.

**2. Decoding is exactly the inverse of encoding.** Both walk the same `d1+d2` positions in the
same order; the encoder multiplies up, the decoder shifts down. Cheap enough to check
*exhaustively* rather than argue about — see below.

**3. A rejected window can never be read as a motif.** Real codes fill `[0, 4**(d1+d2))` and the
sentinel is `4**(d1+d2)` exactly, so the single test `code < 4**(d1+d2)` separates them with no
gap and no overlap. This is why the sentinel is that value and not, say, `-1`.

**4. The arithmetic stays in range.** Codes are `int32`, which holds 15 defined bases; the
shipped patterns use at most 8. Beyond 15 the numeral would silently wrap and every count for
that pattern would be quietly wrong, so `_pattern_window_codes` checks and raises. On numpy ≥ 2
the sentinel assignment happens to raise anyway, but that is a numpy version's behaviour, not a
guarantee this package can make — hence the explicit check.

Properties 1–3 are proved by construction and verified exhaustively here. What none of them
covers is whether the *whole scan* means the same thing as the readable implementation. Nothing
here can establish that -- it is the job of `_check_same_scores` below, which runs both on the
real library and compares every column.

In [10]:
#| hide
# 1 & 2: exhaustive round-trip over the entire code space of several patterns.
for _pat in [(1, 0, 1), (6, 0, 0), (3, 4, 4), (4, 8, 4)]:
    _d1, _sp, _d2 = _pat
    _n = 4 ** (_d1 + _d2)
    _strs = decode_motif_codes(np.arange(_n), _pat)
    test_eq(len(set(_strs)), _n)                                   # 1. injective
    for _s in _strs:                                               # spacer is always N
        test_eq(_s[_d1:_d1 + _sp], "N" * _sp)
    _back = np.zeros(_n, dtype=np.int64)                           # 2. re-encode by hand
    for _pos in list(range(_d1)) + list(range(_d1 + _sp, _d1 + _sp + _d2)):
        _back = _back * 4 + np.array([_B[_s[_pos]] for _s in _strs])
    test_eq(_back.tolist(), list(range(_n)))
print("round-trip exact over every code of", [(1, 0, 1), (6, 0, 0), (3, 4, 4), (4, 8, 4)])

# 3: the sentinel sits exactly one past the top real code, and only bad windows get it.
for _pat in [(6, 0, 0), (3, 4, 4)]:
    _w = _pat[0] + _pat[1] + _pat[2]
    test_eq(int(_pattern_window_codes(encode_library(["N" * _w])[0], _pat)[0, 0]),
            4 ** (_pat[0] + _pat[2]))
    test_eq(int(_pattern_window_codes(encode_library(["A" * _w])[0], _pat)[0, 0]), 0)

# 4: out-of-range patterns are refused rather than wrapped.
_fwd = encode_library(["ACGT" * 20])[0]
_pattern_window_codes(_fwd, (15, 0, 0))                            # fits
test_fail(lambda: _pattern_window_codes(_fwd, (16, 0, 0)), contains="overflow")

round-trip exact over every code of [(1, 0, 1), (6, 0, 0), (3, 4, 4), (4, 8, 4)]


### Encoding a motif back to its code

`encode_motif` is the inverse of `decode_motif_codes`: it turns a motif string back into the
integer the scan counts under. It returns `None` for anything it cannot represent — an
IUPAC-degenerate motif has no single code, and nor has a motif that does not fit the pattern — so
a caller can fall back to a regex rather than silently mis-assign sequences. That fallback is why
`tests/test_vectorized_presence.py` compares the two element for element, not just on summary
statistics.

In [11]:
#| export
def encode_motif(motif, pattern):
    """Integer code of `motif` under `pattern`, inverse of `decode_motif_codes`.

    Returns None if the motif does not fit the pattern or contains a base outside ACGT --
    IUPAC-degenerate motifs have no single code and fall back to the regex path.
    """
    d1, spacer, d2 = pattern
    if len(motif) != d1 + spacer + d2:
        return None
    defined = motif[:d1] + motif[d1 + spacer:]
    if d1 + spacer + d2 and any(c != "N" for c in motif[d1:d1 + spacer]):
        return None                       # spacer must be undefined
    code = 0
    for ch in defined:
        v = "ACGT".find(ch)
        if v < 0:
            return None
        code = code * 4 + v
    return code

In [12]:
#| hide
test_eq(encode_motif("CNNGT", (1, 2, 2)), int("123", 4))   # the decode example, inverted
for _c in range(4 ** 3):                                  # exact inverse over the whole space
    test_eq(encode_motif(decode_motif_codes([_c], (1, 2, 2))[0], (1, 2, 2)), _c)
test_eq(encode_motif("YNNGT", (1, 2, 2)), None)          # IUPAC has no single code
test_eq(encode_motif("CNNG", (1, 2, 2)), None)           # wrong length for the pattern
test_eq(encode_motif("CAAGT", (1, 2, 2)), None)          # spacer must be undefined

### The two strands of one site

An enzyme cuts a double-stranded site, so `GAGACC` and `GGTCTC` are one motif seen from two
sides, and the scan must not count them as two. *Where* they can be folded together is decided by
the pattern, because a motif of `(d1, spacer, d2)` reverse-complements into one of
`(d2, spacer, d1)`:

- when that is the **same** shape — a contiguous motif (`spacer == 0`), or a balanced one
  (`d1 == d2`) — both spellings are codes in the *same* space, and one physical site is found
  twice, once per strand. These fold.
- otherwise the rc spelling lives in **another pattern's** space. Nothing to fold: scanning
  `(4, 4, 3)` on both strands already finds each site exactly once, because a forward
  `(3, 4, 4)` site is picked up on the reverse strand and spelled as `(4, 4, 3)`. This is why
  `find_restricted_motifs` rewrites every pattern to its `canonical_pattern` and drops
  duplicates — searching both shapes would re-find the same sites in mirror spelling, at twice
  the cost.

Folding is not optional arithmetic: applying it to an asymmetric pattern would merge two
*unrelated* motifs, since reversing the digits of a `(4, 4, 3)` code reads the `d2` bases where
the `d1` bases belong. `rc_code_table` therefore refuses any pattern that is not self-rc.

The table itself is one observation: a code is the defined bases as a base-4 numeral, and the
defined bases of the reverse complement are those digits **reversed and complemented**
(`A<->T, C<->G` is `3 - digit`). The spacer is undefined on both strands, so it plays no part.
Because each base is two bits, both operations are bit arithmetic on the code: complementing
every digit is a single XOR with `11` repeated `k` times, and reversing the digits is reversing
the order of the 2-bit groups. `rc_code_table` builds the table that way, and the test below
checks it against `revcomp` on the motif strings over entire code spaces.

In [13]:
#| export
def pattern_is_self_rc(pattern):
    """Does a motif of `pattern` reverse-complement into another motif of the same pattern?

    A motif of `(d1, spacer, d2)` has its reverse complement in `(d2, spacer, d1)`, so the two
    spellings share a code space only when that swap changes nothing: a contiguous motif
    (`spacer == 0`) or a balanced one (`d1 == d2`). This is a property of the shape alone, so it
    holds for any pattern a caller supplies, not just the ones `get_patterns()` returns.
    """
    d1, spacer, d2 = pattern
    return spacer == 0 or d1 == d2


@functools.lru_cache(maxsize=None)
def rc_code_table(pattern):
    """`table[code]` is the code of the reverse-complement motif, for a self-rc `pattern`.

    A code is the motif's defined bases read as a base-4 numeral, and the defined bases of the
    reverse complement are those digits reversed and complemented -- with `A=0 C=1 G=2 T=3`,
    complementing a base is `3 - code`. The spacer is undefined on both strands and plays no
    part.

    Raises for a pattern whose reverse complement has a different shape: there the rc motif is
    not in this code space at all, and folding on this table would merge unrelated motifs.
    """
    if not pattern_is_self_rc(pattern):
        raise ValueError(
            f"pattern {pattern} is not self-reverse-complementary: a motif of it reverse-"
            f"complements into pattern {(pattern[2], pattern[1], pattern[0])}, which is a "
            f"different code space. There is nothing to fold.")
    k = pattern[0] + pattern[2]
    # Each base is a 2-bit group (A=00 C=01 G=10 T=11), so both halves of "reverse complement"
    # are bit operations on the code itself. Reverse: swap the 2-bit groups end for end, done
    # here as a fixed reversal of all sixteen groups of a 32-bit word followed by a shift that
    # drops the 16 - k unused ones. Complement: 3 - d for every digit, which on two bits is
    # d XOR 11, so the whole code complements as one XOR with k groups of 11.
    x = np.arange(4 ** k, dtype=np.uint32)
    x = ((x >> 2) & 0x33333333) | ((x & 0x33333333) << 2)    # swap neighbouring groups
    x = ((x >> 4) & 0x0F0F0F0F) | ((x & 0x0F0F0F0F) << 4)    # swap neighbouring pairs
    x = ((x >> 8) & 0x00FF00FF) | ((x & 0x00FF00FF) << 8)    # swap neighbouring bytes
    x = (x >> 16) | (x << 16)                                # swap the two halves
    table = ((x >> np.uint32(32 - 2 * k)) ^ np.uint32(4 ** k - 1)).astype(np.int64)
    # A lookup into this, rather than evaluating the formula on every window, because in numpy
    # each line above is a full pass over the array: on a 30k x 300 window-code matrix the
    # formula costs 60 ms per pattern and the lookup 21 ms. The table is 4**k entries, the same
    # size as the bincount the scan already allocates for this pattern.
    table.setflags(write=False)           # cached and shared; never mutate in place
    return table

In [14]:
#| hide
# the shape rule
test_eq(pattern_is_self_rc((6, 0, 0)), True)     # contiguous: rc of a 6-mer is a 6-mer
test_eq(pattern_is_self_rc((4, 4, 4)), True)     # balanced
test_eq(pattern_is_self_rc((4, 4, 3)), False)    # rc has shape (3, 4, 4)
test_fail(lambda: rc_code_table((4, 4, 3)), contains="nothing to fold")

# the table agrees with revcomp on the string, over whole code spaces
from randseq.utils import revcomp
for _pat in [(1, 0, 1), (5, 0, 0), (6, 0, 0), (3, 4, 3), (4, 2, 4)]:
    _n = 4 ** (_pat[0] + _pat[2])
    _strs = decode_motif_codes(np.arange(_n), _pat)
    _tbl = rc_code_table(_pat)
    test_eq([revcomp(_s) for _s in _strs], list(decode_motif_codes(_tbl, _pat)))
    test_eq(_tbl[_tbl].tolist(), list(range(_n)))        # rc is an involution
print("rc table == revcomp over every code of", [(1, 0, 1), (5, 0, 0), (6, 0, 0), (3, 4, 3), (4, 2, 4)])

# folding is exactly 'half the space, plus the palindromes'
for _pat in [(6, 0, 0), (3, 4, 3)]:
    _n = 4 ** (_pat[0] + _pat[2])
    _tbl = rc_code_table(_pat)
    _folded = np.minimum(np.arange(_n), _tbl)
    _pal = int((_tbl == np.arange(_n)).sum())
    test_eq(len(set(_folded.tolist())), (_n + _pal) // 2)

rc table == revcomp over every code of [(1, 0, 1), (5, 0, 0), (6, 0, 0), (3, 4, 3), (4, 2, 4)]


## Reading motif presence off the window codes

Scoring a pattern asks what motifs a sequence contains. The rest of the pipeline asks the reverse,
of a handful of specific candidates: **which sequences contain this motif, on either strand?**

Done directly that is a regex per sequence per motif, and it is the slow half of the pipeline — a
strain with no real system is the worst case, because it carries the most candidates. The window
codes are already computed, so presence is a **lookup in that array**, done once per pattern
rather than once per motif: 0.7 s instead of 4.6 s on the shipped panel.

`motif_presence_matrix` returns `(matrix, handled)`. A motif the fast path cannot encode —
IUPAC-degenerate, or of an unknown pattern — is left as an all-`False` column and flagged in
`handled`, so the caller falls back to the regex rather than reading a blank column as "absent".

In [15]:
#| export
def motif_presence_matrix(encoded_library, motifs, patterns):
    """Boolean (n_sequences, n_motifs): does sequence i contain motif j, on either strand?

    One pass per distinct pattern rather than one per motif, and a lookup into the window codes
    rather than a regex per sequence. Motifs whose pattern is unknown, or which contain IUPAC
    ambiguity codes, are left as all-False here; the caller falls back to the regex path for
    those, so behaviour is unchanged and only the common case gets faster.

    Returns `(matrix, handled)` where `handled` marks the motifs the matrix actually covers.
    """
    fwd, rc = encoded_library
    n_seq = fwd.shape[0]
    matrix = np.zeros((n_seq, len(motifs)), dtype=bool)
    handled = np.zeros(len(motifs), dtype=bool)

    by_pattern = defaultdict(list)
    for j, (motif, pattern) in enumerate(zip(motifs, patterns)):
        if pattern is None:
            continue
        code = encode_motif(motif, pattern)
        if code is not None:
            by_pattern[pattern].append((j, code))

    for pattern, entries in by_pattern.items():
        n_codes = 4 ** (pattern[0] + pattern[2])
        # code -> column in `matrix`, or -1 for codes we do not care about
        lookup = np.full(n_codes + 1, -1, dtype=np.int32)
        for j, code in entries:
            lookup[code] = j
        codes = np.concatenate(
            [_pattern_window_codes(fwd, pattern), _pattern_window_codes(rc, pattern)], axis=1)
        cols = lookup[codes]
        rows, where = np.nonzero(cols >= 0)
        if len(rows):
            matrix[rows, cols[rows, where]] = True
        for j, _ in entries:
            handled[j] = True

    return matrix, handled

In [16]:
#| hide
_enc = encode_library(["GAGACCTT", "AAGAGACC", "TTTTTTTT"])
_m, _h = motif_presence_matrix(_enc, ["GAGACC", "TTTTTT"], [(6, 0, 0), (6, 0, 0)])
assert _h.all()
test_eq(_m[:, 0].tolist(), [True, True, False])
test_eq(_m[:, 1].tolist(), [False, False, True])
# both strands are searched, so the reverse complement finds the same sequences
_m2, _ = motif_presence_matrix(_enc, ["GGTCTC"], [(6, 0, 0)])
test_eq(_m2[:, 0].tolist(), [True, True, False])
# what the fast path cannot encode is reported as unhandled and left False, NOT as absent --
# the caller falls back to the regex, so a silent wrong answer is impossible
_m3, _h3 = motif_presence_matrix(_enc, ["GAGAYC"], [(6, 0, 0)])
test_eq(_h3.tolist(), [False])
test_eq(int(_m3.sum()), 0)

In [17]:
_seqs = ["GAGACCTT", "AAGAGACC", "TTTTTTTT", "GGGGGGGG"]
_m, _handled = motif_presence_matrix(encode_library(_seqs), ["GAGACC", "TTTTTT"],
                                     [(6, 0, 0), (6, 0, 0)])
print(f"matrix {_m.shape} of {_m.dtype}, handled={_handled}")
pd.DataFrame(_m, index=_seqs, columns=["GAGACC", "TTTTTT"])

matrix (4, 2) of bool, handled=[ True  True]


,GAGACC,TTTTTT
GAGACCTT,True,False
AAGAGACC,True,False
TTTTTTTT,False,True
GGGGGGGG,False,False


One row per sequence, one column per motif. `GAGACC` is present in the first two sequences — in the
second it is read on the other strand, as `GGTCTC` — and `GGGGGGGG` carries neither.

In [16]:
#| hide
import nbdev; nbdev.nbdev_export()